---

## **DIPLOME UNIVERSITAIRE SDA**
---
## **Projet Generative AI — Assistant Intelligent RAG Catastrophes Climatiques**

---

*Professeur* : Kamila Kare

*Étudiants* : Diego Merchán, Camille Koenig, Jayson Phan Nguyen, Xia Bizot

Promotion 007

Avril 2026

---

**Corpus** : 10 rapports scientifiques (GIEC AR6, Copernicus, EM-DAT, NOAA, JRC, WMO, EU CELEX)

**Repo** : https://github.com/diegomerchanm/catastrophes-climatiques-rag

---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans recalculer si les fichiers existent déjà)
- **traçable** (quality gates go/no-go explicites)

**Convention :** chaque cellule de code doit produire une sortie visible.

---

---

## Contexte du projet

**Architecture :**
```
Question utilisateur
    ↓
Agent Agentic RAG (ReAct — LangGraph)
    ↓ décide seul quels outils appeler
    ├── search_corpus (RAG hybride BM25 + Dense + reranking)
    ├── get_weather / get_historical_weather / get_forecast (OpenMeteo)
    ├── web_search (Tavily + DuckDuckGo fallback)
    ├── calculator
    └── send_email
    ↓ boucle Reason → Act → Observe → Repeat
Réponse argumentée avec sources
```

**Ce notebook** démontre la capacité prédictive du système : croisement automatique corpus GIEC + données météo pour produire des analyses de risque climatique.

---

# NOTEBOOK 5 — Analyse de Risque Prédictive

**Auteur :** Xia Bizot

---

### Objectif

Démontrer que le système croise automatiquement les données du corpus scientifique (GIEC, Copernicus, EM-DAT) avec les données météo (historiques et prévisions) pour produire des analyses de risque argumentées, quantifiées et sourcées. Passé, présent, futur.

---

### Plan du notebook

| Section | Contenu |
|---|---|
| 1. Configuration | Imports, chemins, seed, constantes, quality gates |
| 2. Analyse du passé | Catastrophes historiques + météo historique croisée |
| 3. Analyse du présent | Conditions actuelles vs seuils critiques GIEC |
| 4. Analyse du futur | Prévisions + seuils + précédents → score de risque |
| 5. Analyse par ville | Comparatif multi-villes |
| 6. Comparaison temporelle | 2022 vs 2023 en Europe |
| 7. Revue de presse | Actualités climatiques web |
| 8. Décisions GO / NO-GO | Scénarios d'aide à la décision |
| 9. Tracking MLflow | Chaque prédiction trackée |
| 10. Résultats | Tableaux, visualisations |
| 11. Conclusions | Synthèse, quality gate |

---

### Décisions prises dans ce notebook

- **Périmètre :** villes méditerranéennes et européennes à risque
- **Stratégie :** l'agent décide seul quels outils enchaîner pour produire l'analyse
- **Output :** NB5_risques.csv, NB5_figures, runs MLflow

---

### Hypothèse testable

> Le système est capable de croiser automatiquement les données du corpus GIEC avec les données météo (historiques et prévisions) pour produire une analyse de risque argumentée et sourcée, sans intervention humaine dans le choix des outils.

---

---

## 1. Configuration

### 1.1. Imports et timing

In [ ]:
import os
import sys
import time
import logging
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

notebook_start_time = time.time()

print('>> 1.1. Imports : OK')

### 1.2. Chemins et environnement

In [ ]:
BASE = Path('..')
OUTPUT_DIR = BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE))

from dotenv import load_dotenv
load_dotenv(BASE / '.env')

print(f'Dossier output : {OUTPUT_DIR.resolve()}')
print('>> 1.2. Chemins : OK')

### 1.3. Versions et seed

In [ ]:
SEED = 42
np.random.seed(SEED)

print(f'Python  : {sys.version}')
print(f'SEED    : {SEED}')
print('>> 1.3. Versions / seed : OK')

### 1.4. Constantes du projet

In [ ]:
from src.config import (
    AGENT_CONFIGS, MODEL_PRICING,
    RETRIEVER_K, BM25_WEIGHT, DENSE_WEIGHT,
)
from src.agents.agent import get_prompt_version

# Villes surveillées pour l'analyse de risque
VILLES_ANALYSE = ['Marseille', 'Nice', 'Lyon', 'Paris', 'Bordeaux']

# Seuils critiques (mm de précipitations sur 24h)
SEUIL_ALERTE = 80
SEUIL_CRITIQUE = 100

print(f'Villes analysées     : {VILLES_ANALYSE}')
print(f'Seuil alerte         : {SEUIL_ALERTE} mm')
print(f'Seuil critique       : {SEUIL_CRITIQUE} mm')
print(f'Prompt version       : {get_prompt_version()}')
print(f'Orchestrateur        : {AGENT_CONFIGS["orchestrator"]["model"]}')
print('>> 1.4. Constantes projet : OK')

### 1.5. Quality gates

In [ ]:
DATA_DIR = BASE / 'data' / 'raw'
FAISS_DIR = BASE / 'faiss_store'
pdfs = list(DATA_DIR.glob('*.pdf'))

checks = {
    'cle_anthropic': (bool(os.getenv('ANTHROPIC_API_KEY')), bool(os.getenv('ANTHROPIC_API_KEY'))),
    'pdfs_corpus': (len(pdfs), len(pdfs) > 0),
    'faiss_store': (FAISS_DIR.exists(), FAISS_DIR.exists()),
}

all_ok = True
for k, (valeur, condition) in checks.items():
    status = '[OK]' if condition else '[KO]'
    if not condition:
        all_ok = False
    print(f'  {status} {k} : {valeur}')

assert all_ok, 'Quality gates KO'
print('>> 1.5. Quality gates : OK')

### 1.6. Initialisation MLflow

In [ ]:
import mlflow

mlflow.set_experiment('nb5-analyse-risque-predictive')
print(f'MLflow version : {mlflow.__version__}')
print(f'Expérience     : nb5-analyse-risque-predictive')
print('>> 1.6. MLflow : OK')

---

## 2. Analyse du PASSÉ

**Scénario :** "Quelles catastrophes climatiques ont touché la Méditerranée en 2023 ?"  
L'agent doit chercher dans le corpus ET vérifier la météo historique des lieux/dates trouvés.

In [ ]:
from src.agents.agent import run_agent, get_token_summary, reset_token_counter

# Question 1 : événements passés
reset_token_counter()
t0 = time.time()

question_passe = 'Quelles catastrophes climatiques majeures ont touché la Méditerranée en 2023 ? Croise avec les données météo historiques des lieux et dates concernés.'
reponse_passe = run_agent(question_passe)

duree_passe = round(time.time() - t0, 2)
tokens_passe = get_token_summary()

print(f'Durée : {duree_passe}s')
print(f'Tokens : {tokens_passe["total_tokens"]}')
print(f'Coût : ${tokens_passe["estimated_cost_usd"]:.4f}')
print(f'\n{"=" * 60}\n')
print(reponse_passe)

# MLflow tracking
with mlflow.start_run(run_name='passe_mediterranee_2023'):
    mlflow.log_param('type_analyse', 'passé')
    mlflow.log_param('question', question_passe[:250])
    mlflow.log_param('region', 'Méditerranée')
    mlflow.log_param('annee', 2023)
    mlflow.log_metric('duree_s', duree_passe)
    mlflow.log_metric('total_tokens', tokens_passe['total_tokens'])
    mlflow.log_metric('cout_usd', tokens_passe['estimated_cost_usd'])
    mlflow.log_metric('longueur_reponse', len(reponse_passe))

print('>> 2. Analyse passé : OK')

### Analyse

**Ce qu'on observe :**
- [L'agent a-t-il appelé search_corpus ET get_historical_weather ?]
- [Les sources sont-elles citées avec [Source, Page] ?]
- [Les données météo historiques corroborent-elles les événements du corpus ?]

**Pourquoi c'est important :**
- Le croisement automatique corpus + météo prouve que l'agent raisonne, pas juste qu'il récite
- Les citations garantissent la traçabilité (faithfulness)

---

---

## 3. Analyse du PRÉSENT

**Scénario :** "Il pleut 120mm à Lyon aujourd'hui, c'est grave ?"  
L'agent doit comparer avec les seuils critiques du corpus GIEC et les événements passés similaires.

In [ ]:
reset_token_counter()
t0 = time.time()

question_present = 'Il pleut 120mm à Lyon aujourd\'hui. Est-ce que c\'est grave ? Compare avec les seuils critiques du GIEC et les événements passés similaires.'
reponse_present = run_agent(question_present)

duree_present = round(time.time() - t0, 2)
tokens_present = get_token_summary()

print(f'Durée : {duree_present}s')
print(f'Tokens : {tokens_present["total_tokens"]}')
print(f'Coût : ${tokens_present["estimated_cost_usd"]:.4f}')
print(f'\n{"=" * 60}\n')
print(reponse_present)

with mlflow.start_run(run_name='present_lyon_120mm'):
    mlflow.log_param('type_analyse', 'présent')
    mlflow.log_param('question', question_present[:250])
    mlflow.log_param('ville', 'Lyon')
    mlflow.log_param('precipitation_mm', 120)
    mlflow.log_metric('duree_s', duree_present)
    mlflow.log_metric('total_tokens', tokens_present['total_tokens'])
    mlflow.log_metric('cout_usd', tokens_present['estimated_cost_usd'])

print('>> 3. Analyse présent : OK')

### Analyse

**Ce qu'on observe :**
- [L'agent a-t-il comparé 120mm avec les seuils GIEC ?]
- [A-t-il trouvé des événements passés similaires dans le corpus ?]
- [Le niveau de risque est-il quantifié (faible/modéré/élevé/critique) ?]

---

---

## 4. Analyse du FUTUR

**Scénario :** "Quel est le risque d'inondation à Marseille cette semaine ?"  
L'agent consulte les prévisions, compare avec les seuils GIEC, et réfère aux précédents historiques.

In [ ]:
reset_token_counter()
t0 = time.time()

question_futur = 'Quel est le risque d\'inondation à Marseille cette semaine ? Consulte les prévisions météo, compare avec les seuils critiques du GIEC, et réfère-toi aux événements passés similaires.'
reponse_futur = run_agent(question_futur)

duree_futur = round(time.time() - t0, 2)
tokens_futur = get_token_summary()

print(f'Durée : {duree_futur}s')
print(f'Tokens : {tokens_futur["total_tokens"]}')
print(f'Coût : ${tokens_futur["estimated_cost_usd"]:.4f}')
print(f'\n{"=" * 60}\n')
print(reponse_futur)

with mlflow.start_run(run_name='futur_marseille_risque'):
    mlflow.log_param('type_analyse', 'futur')
    mlflow.log_param('question', question_futur[:250])
    mlflow.log_param('ville', 'Marseille')
    mlflow.log_metric('duree_s', duree_futur)
    mlflow.log_metric('total_tokens', tokens_futur['total_tokens'])
    mlflow.log_metric('cout_usd', tokens_futur['estimated_cost_usd'])

print('>> 4. Analyse futur : OK')

### Analyse

**Ce qu'on observe :**
- [L'agent a-t-il enchaîné get_forecast → search_corpus → analyse croisée ?]
- [Le risque est-il quantifié et qualifié ?]
- [Y a-t-il des recommandations concrètes ?]

**Pourquoi c'est le test le plus important :**
- C'est de la **prédiction** : on anticipe un risque futur basé sur des données scientifiques
- C'est la valeur ajoutée du système par rapport à une simple appli météo

---

---

## 5. Analyse par ville

Comparer le risque sur plusieurs villes simultanément.

In [ ]:
results_villes = []

for ville in VILLES_ANALYSE:
    reset_token_counter()
    t0 = time.time()
    
    question = f'Quel est le niveau de risque climatique à {ville} cette semaine ? Donne un score : faible, modéré, élevé ou critique, avec justification.'
    reponse = run_agent(question)
    
    duree = round(time.time() - t0, 2)
    tokens = get_token_summary()
    
    # Détecter le niveau de risque dans la réponse
    reponse_lower = reponse.lower()
    if 'critique' in reponse_lower:
        niveau = 'CRITIQUE'
    elif 'élevé' in reponse_lower or 'eleve' in reponse_lower:
        niveau = 'ÉLEVÉ'
    elif 'modéré' in reponse_lower or 'modere' in reponse_lower:
        niveau = 'MODÉRÉ'
    else:
        niveau = 'FAIBLE'
    
    results_villes.append({
        'ville': ville,
        'niveau_risque': niveau,
        'tokens': tokens['total_tokens'],
        'cout_usd': tokens['estimated_cost_usd'],
        'duree_s': duree,
        'longueur_reponse': len(reponse),
    })
    
    print(f'  {ville:15s} → {niveau:10s} ({tokens["total_tokens"]} tokens, {duree}s)')
    
    # MLflow
    with mlflow.start_run(run_name=f'risque_{ville.lower()}'):
        mlflow.log_param('ville', ville)
        mlflow.log_param('niveau_risque', niveau)
        mlflow.log_metric('duree_s', duree)
        mlflow.log_metric('total_tokens', tokens['total_tokens'])
        mlflow.log_metric('cout_usd', tokens['estimated_cost_usd'])

df_villes = pd.DataFrame(results_villes)
print(f'\n{df_villes.to_string()}')
print('>> 5. Analyse par ville : OK')

In [ ]:
# Visualisation
couleurs_risque = {
    'FAIBLE': '#2ecc71',
    'MODÉRÉ': '#f39c12',
    'ÉLEVÉ': '#e74c3c',
    'CRITIQUE': '#8e44ad',
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = [couleurs_risque.get(n, 'steelblue') for n in df_villes['niveau_risque']]
df_villes.plot(x='ville', y='duree_s', kind='bar', ax=ax, color=colors, legend=False)
ax.set_title('Analyse de risque par ville — Durée et niveau de risque')
ax.set_xlabel('Ville')
ax.set_ylabel('Durée (s)')

# Ajouter les labels de risque
for i, (_, row) in enumerate(df_villes.iterrows()):
    ax.text(i, row['duree_s'] + 0.2, row['niveau_risque'], ha='center', fontsize=9, fontweight='bold')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'NB5_risque_par_ville.png', dpi=200, bbox_inches='tight')
print(f'  [OK] NB5_risque_par_ville.png sauvegardé')
plt.show()

### Analyse

**Ce qu'on observe :**
- [Quelles villes sont les plus à risque ?]
- [Les niveaux de risque sont-ils cohérents avec la géographie (Méditerranée vs Atlantique) ?]
- [Les villes côtières sont-elles plus à risque que les villes intérieures ?]

---

---

## 6. Comparaison temporelle

In [ ]:
reset_token_counter()
t0 = time.time()

question_comparaison = 'Compare les catastrophes climatiques en Europe entre 2022 et 2023. Quelles tendances observes-tu ? Utilise le corpus et les données météo historiques.'
reponse_comparaison = run_agent(question_comparaison)

duree_comp = round(time.time() - t0, 2)
tokens_comp = get_token_summary()

print(f'Durée : {duree_comp}s | Tokens : {tokens_comp["total_tokens"]} | Coût : ${tokens_comp["estimated_cost_usd"]:.4f}')
print(f'\n{"=" * 60}\n')
print(reponse_comparaison)
print('>> 6. Comparaison temporelle : OK')

---

## 7. Revue de presse web

In [ ]:
reset_token_counter()
t0 = time.time()

question_web = 'Quelles sont les dernières catastrophes climatiques en Europe cette semaine ? Cherche sur le web et croise avec le corpus scientifique.'
reponse_web = run_agent(question_web)

duree_web = round(time.time() - t0, 2)
tokens_web = get_token_summary()

print(f'Durée : {duree_web}s | Tokens : {tokens_web["total_tokens"]} | Coût : ${tokens_web["estimated_cost_usd"]:.4f}')
print(f'\n{"=" * 60}\n')
print(reponse_web)
print('>> 7. Revue de presse : OK')

---

## 8. Décisions GO / NO-GO

Scénarios d'aide à la décision concrets.

In [ ]:
scenarios_decision = [
    'Je veux organiser un événement en extérieur à Marseille samedi prochain. Est-ce risqué ?',
    'Je suis assureur et je dois évaluer si les inondations de 2023 à Nice étaient prévisibles. Donne-moi des preuves.',
    'Je suis maire de Lyon. Dois-je lancer une évacuation préventive cette semaine ?',
]

results_decisions = []

for i, scenario in enumerate(scenarios_decision):
    reset_token_counter()
    t0 = time.time()
    reponse = run_agent(scenario)
    duree = round(time.time() - t0, 2)
    tokens = get_token_summary()
    
    print(f'\n{"=" * 60}')
    print(f'Scénario {i+1} : {scenario[:60]}...')
    print(f'Durée : {duree}s | Tokens : {tokens["total_tokens"]}')
    print(f'{"=" * 60}')
    print(reponse[:500])
    print('...' if len(reponse) > 500 else '')
    
    results_decisions.append({
        'scenario': scenario[:60],
        'tokens': tokens['total_tokens'],
        'cout_usd': tokens['estimated_cost_usd'],
        'duree_s': duree,
    })

df_decisions = pd.DataFrame(results_decisions)
print(f'\n{df_decisions.to_string()}')
print('>> 8. Décisions GO/NO-GO : OK')

### Analyse

**Cibles utilisateurs démontrées :**
- **Collectivités locales** : évacuation préventive (maire de Lyon)
- **Assureurs** : preuve de prévisibilité des sinistres
- **Entreprises** : planification événementielle

**Ce qu'on observe :**
- [L'agent donne-t-il une recommandation GO / NO-GO claire ?]
- [Les preuves sont-elles sourcées ?]

---

---

## 9. Tracking MLflow — Dashboard

In [ ]:
# Lister tous les runs de ce notebook
try:
    runs = mlflow.search_runs(experiment_names=['nb5-analyse-risque-predictive'], max_results=20)
    if len(runs) > 0:
        cols = [c for c in ['params.type_analyse', 'params.ville', 'params.niveau_risque', 
                            'metrics.total_tokens', 'metrics.cout_usd', 'metrics.duree_s'] if c in runs.columns]
        print(runs[cols].to_string())
    else:
        print('Aucun run MLflow trouvé.')
except Exception as e:
    print(f'Erreur MLflow : {e}')

print('\nPour visualiser le dashboard MLflow :')
print('  mlflow ui --host 127.0.0.1 --port 8080')
print('  Ouvrir http://localhost:8080')
print('>> 9. MLflow tracking : OK')

---

## 10. Résultats

### 10.1. Tableau synthétique

In [ ]:
# Récapitulatif de toutes les analyses
recap = [
    {'analyse': 'Passé — Méditerranée 2023', 'tokens': tokens_passe['total_tokens'], 'cout': tokens_passe['estimated_cost_usd'], 'duree': duree_passe},
    {'analyse': 'Présent — Lyon 120mm', 'tokens': tokens_present['total_tokens'], 'cout': tokens_present['estimated_cost_usd'], 'duree': duree_present},
    {'analyse': 'Futur — Marseille risque', 'tokens': tokens_futur['total_tokens'], 'cout': tokens_futur['estimated_cost_usd'], 'duree': duree_futur},
    {'analyse': 'Comparaison 2022 vs 2023', 'tokens': tokens_comp['total_tokens'], 'cout': tokens_comp['estimated_cost_usd'], 'duree': duree_comp},
    {'analyse': 'Revue de presse web', 'tokens': tokens_web['total_tokens'], 'cout': tokens_web['estimated_cost_usd'], 'duree': duree_web},
]

df_recap = pd.DataFrame(recap)
csv_path = OUTPUT_DIR / 'NB5_analyse_risque_results.csv'
df_recap.to_csv(csv_path, index=False)
assert csv_path.exists()
print(f'  [OK] {csv_path.name} sauvegardé')
print()
print(df_recap.to_string())
print(f'\nTotal tokens : {df_recap["tokens"].sum()}')
print(f'Coût total   : ${df_recap["cout"].sum():.4f}')

In [ ]:
# Sauvegarde résultats par ville
csv_villes = OUTPUT_DIR / 'NB5_risque_par_ville.csv'
df_villes.to_csv(csv_villes, index=False)
assert csv_villes.exists()
print(f'  [OK] {csv_villes.name} sauvegardé')

---

## 11. Conclusions

### Synthèse

[À compléter après exécution — résumer les résultats clés des analyses passé/présent/futur.]

---

### Quality gate finale

| Constat | Preuve | Décision |
|---|---|---|
| L'agent croise corpus + météo automatiquement | Réponses sections 2-4 | Hypothèse validée |
| Le risque est quantifié par ville | Tableau section 5 | Système opérationnel |
| Les décisions GO/NO-GO sont argumentées | Section 8 | Valeur métier démontrée |
| MLflow tracke chaque prédiction | Dashboard section 9 | Traçabilité garantie |

---

### Hypothèse : [validée / invalidée]

[À compléter — le système a-t-il croisé automatiquement les sources sans intervention ?]

---

### Limites et avertissement scientifique

**Ce système est un outil d'aide à la décision, pas un modèle prédictif validé scientifiquement.**

Il croise des sources fiables (GIEC, OpenMeteo) mais le raisonnement est produit par un LLM qui peut halluciner. En production, il devrait être :
- **Validé par un expert climatologue** avant toute utilisation opérationnelle
- **Backtesté sur des événements passés** pour mesurer la fiabilité des prédictions
- **Complété par des modèles physiques** de simulation atmosphérique (Météo France, ECMWF) que notre système ne remplace pas

**Limites techniques :**
- Ce n'est **pas du ML prédictif classique** (pas de modèle entraîné sur un dataset de catastrophes). C'est du **raisonnement par analogie et seuils**, produit par un LLM
- Les seuils critiques sont extraits du corpus par le LLM (pas codés en dur) — dépend de la qualité du RAG et de la faithfulness du LLM
- Le score de risque est qualitatif (faible/modéré/élevé/critique), pas un score statistique calibré
- Les prévisions météo sont limitées à 7 jours (contrainte OpenMeteo) — au-delà la fiabilité chute fortement
- Pas de validation a posteriori (le risque prédit s'est-il réalisé ?)
- Le LLM peut halluciner des seuils, des dates ou des événements non documentés dans le corpus
- Le corpus est figé (10 PDFs) — les événements postérieurs à la publication des rapports ne sont pas couverts

**Ce qu'on peut affirmer :**
- Les sources sont fiables (GIEC, Copernicus, EM-DAT — publications scientifiques peer-reviewed)
- Les données météo sont factuelles (OpenMeteo — données réelles des stations)
- Le croisement est tracé et sourcé (citations [Source, Page], logging, MLflow)
- Le système est transparent sur son raisonnement (boucle ReAct observable)

---

### Axes d'amélioration

- **Validation scientifique** : backtester les prédictions sur des événements passés connus (le système aurait-il prédit les inondations de 2023 ?)
- Score de risque chiffré basé sur le ratio précipitations / seuil critique
- Auto-évaluation des prédictions (vérifier le vendredi si l'alerte du lundi était juste)
- Alertes automatiques par email quand le risque est élevé/critique (via send_email)
- Intégration Météo France pour les données officielles françaises
- Enrichissement du corpus avec les rapports les plus récents (upload PDF dynamique)
- Détection d'hallucinations : vérifier que les seuils cités par le LLM existent vraiment dans le corpus

In [ ]:
elapsed = round(time.time() - notebook_start_time, 2)
print(f'Temps total du notebook : {elapsed}s')
print('>> NOTEBOOK 5 TERMINÉ')